# Google Translate (deep-translator): Inference & Evaluation on Annotated Filtered Dataset
This notebook performs machine translation inference on `Annotated Data - Filtered Dataset.csv` using **Google Translate** (via `deep-translator` en -> ml) and computes the complete evaluation metric suite (SacreBLEU, Indic-Tokenized BLEU, chrF, chrF++, METEOR, and TER), directly comparable to IndicTrans2 and BhashaVerse.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install deep-translator and evaluation packages
%%capture
!pip install -q deep-translator sacrebleu evaluate indic-nlp-library
!python3 -c "import nltk; nltk.download('punkt'); nltk.download('wordnet'); nltk.download('omw-1.4')"

## 1. Load & Preprocess `Annotated Data - Filtered Dataset.csv`

In [ ]:
import os
import pandas as pd

# Automatically locate the uploaded or Drive dataset
CANDIDATE_PATHS = [
    "/content/Annotated Data - Filtered Dataset.csv",
    "/content/drive/MyDrive/Annotated Data - Filtered Dataset.csv",
    "/content/drive/MyDrive/Annotated_Data_Filtered_Dataset.csv",
    "Annotated Data - Filtered Dataset.csv",
]

CSV_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH is None:
    CSV_PATH = "/content/Annotated Data - Filtered Dataset.csv"
    print(f"Warning: Dataset not found automatically. Expected at: {CSV_PATH}")
    print("Please upload 'Annotated Data - Filtered Dataset.csv' to /content or Google Drive.")
else:
    print(f"Successfully located dataset at: {CSV_PATH}")

# Read and clean dataset
df = pd.read_csv(CSV_PATH)
initial_len = len(df)

# Drop any row where English Sentence is NaN or blank (e.g. trailing row 100)
df = df.dropna(subset=["English Sentence"]).copy()
df["English Sentence"] = df["English Sentence"].astype(str).str.strip()
df = df[df["English Sentence"] != ""].reset_index(drop=True)

# Ensure reference sentences are stripped
df["Malayalam Sentence"] = df["Malayalam Sentence"].astype(str).str.strip()

# Capture alternative canonical reference from 'Unnamed: 4' if available
if "Unnamed: 4" in df.columns:
    df["Alternative Malayalam"] = df["Unnamed: 4"].fillna("").astype(str).str.strip()
else:
    df["Alternative Malayalam"] = ""

en_sents = df["English Sentence"].tolist()
reference = df["Malayalam Sentence"].tolist()

print(f"Total valid sentences loaded: {len(en_sents)} (filtered from {initial_len} rows).")
print("\nSample Pair [0]:")
print("English  :", en_sents[0])
print("Malayalam:", reference[0])

## 2. Google Translate Inference via `deep-translator`
Translates English sentences to Malayalam (`en` -> `ml`) in batches with rate-limiting pauses and checkpointing.

In [ ]:
import os
import time
import random
import json
import urllib.request
import urllib.parse
import pandas as pd
from tqdm.auto import tqdm

# Optional fallback imports
try:
    from deep_translator import GoogleTranslator, MyMemoryTranslator
except ImportError:
    GoogleTranslator, MyMemoryTranslator = None, None

SAVE_DIR = "/content/drive/MyDrive/GoogleTranslate_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "google_translate_checkpoint.csv")

BATCH_SIZE = 5
MAX_RETRIES = 6

# Rotating desktop User-Agents to prevent bot detection
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:125.0) Gecko/20100101 Firefox/125.0",
]

def _translate_google_api(text, src="en", tgt="ml", timeout=12):
    """
    Direct client API endpoint used by Google Chrome / Mobile apps.
    Returns structured JSON directly without HTML scraping, completely bypassing
    the translate.google.com/m web scraping 429 rate limit in Colab.
    """
    url = f"https://translate.googleapis.com/translate_a/single?client=gtx&sl={src}&tl={tgt}&dt=t&q={urllib.parse.quote(text)}"
    headers = {"User-Agent": random.choice(USER_AGENTS)}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    parts = [part[0] for part in data[0] if part and part[0]]
    return "".join(parts).strip()

def _translate_deep_translator(text, src="en", tgt="ml"):
    """Fallback via deep_translator.GoogleTranslator."""
    if GoogleTranslator is None:
        raise RuntimeError("deep-translator not available")
    return GoogleTranslator(source=src, target=tgt).translate(text)

def _translate_mymemory(text, src="en", tgt="ml"):
    """Fallback via deep_translator.MyMemoryTranslator."""
    if MyMemoryTranslator is None:
        raise RuntimeError("MyMemoryTranslator not available")
    return MyMemoryTranslator(source=src, target=tgt).translate(text)

def translate_with_retry(text, src="en", tgt="ml"):
    """
    Multi-tier resilient translation engine with automatic fallbacks and exponential backoff:
    Tier 1: Direct Google Translate Client API (fastest, JSON-based, no HTML 429 scraping block)
    Tier 2: deep_translator.GoogleTranslator
    Tier 3: deep_translator.MyMemoryTranslator
    """
    if not text or not str(text).strip():
        return ""
    text = str(text).strip()

    for attempt in range(MAX_RETRIES):
        # Tier 1: Direct Google Client API
        try:
            res = _translate_google_api(text, src=src, tgt=tgt)
            if res and len(res) > 0:
                time.sleep(random.uniform(0.1, 0.25))  # Polite jitter
                return res
        except Exception:
            pass

        # Tier 2: deep_translator Google
        try:
            res = _translate_deep_translator(text, src=src, tgt=tgt)
            if res and len(res) > 0:
                time.sleep(random.uniform(0.15, 0.3))
                return res
        except Exception:
            pass

        # Tier 3: deep_translator MyMemory
        try:
            res = _translate_mymemory(text, src=src, tgt=tgt)
            if res and len(res) > 0:
                time.sleep(random.uniform(0.15, 0.3))
                return res
        except Exception:
            pass

        # Exponential backoff if all 3 tiers temporarily hit a snag
        wait_time = min(15.0, (1.5 ** attempt) + random.uniform(0.5, 1.0))
        print(f"\n[Retry {attempt + 1}/{MAX_RETRIES}] Endpoints busy, pausing {wait_time:.1f}s...")
        time.sleep(wait_time)

    raise RuntimeError(f"Failed to translate sentence after {MAX_RETRIES} attempts: {text[:40]}...")

# ---------------------------------------------------------
# Load Checkpoint (Resume-Safe)
# ---------------------------------------------------------
predictions = []
if os.path.exists(CHECKPOINT_PATH):
    try:
        ckpt_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
        if "google_translate_malayalam" in ckpt_df.columns:
            existing = ckpt_df["google_translate_malayalam"].tolist()
            while existing and pd.isna(existing[-1]):
                existing.pop()
            predictions = existing
            print(f"Resuming from checkpoint: {len(predictions)}/{len(en_sents)} translated.")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")

start_index = len(predictions)
print(f"Translating sentences {start_index} to {len(en_sents)} (English -> Malayalam)...")

# ---------------------------------------------------------
# Batch Translation with Live Progress
# ---------------------------------------------------------
for i in tqdm(range(start_index, len(en_sents), BATCH_SIZE), desc="Google Translating (En -> Ml)"):
    batch = en_sents[i : i + BATCH_SIZE]
    batch_preds = []
    for text in batch:
        pred = translate_with_retry(text)
        batch_preds.append(pred)
    predictions.extend(batch_preds)

    # Save checkpoint after every batch
    temp_df = df.iloc[: len(predictions)].copy()
    temp_df["google_translate_malayalam"] = predictions
    temp_df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

print("\nAll sentences translated successfully!")
assert len(predictions) == len(en_sents), f"Mismatch: {len(en_sents)} inputs vs {len(predictions)} outputs"

# ---------------------------------------------------------
# Save Final Predictions to Drive and Local
# ---------------------------------------------------------
df["google_translate_malayalam"] = predictions

final_csv_drive = os.path.join(SAVE_DIR, "google_translate_annotated_predictions.csv")
final_csv_local = "/content/google_translate_annotated_predictions.csv"

df.to_csv(final_csv_drive, index=False, encoding="utf-8-sig")
df.to_csv(final_csv_local, index=False, encoding="utf-8-sig")

print("\nFinal predictions saved successfully:")
print(f"1. Drive: {final_csv_drive}")
print(f"2. Local: {final_csv_local}")

display(df[["English Sentence", "Malayalam Sentence", "google_translate_malayalam"]].head(5))


## 3. Comprehensive & Rigorous Evaluation
Evaluating BLEU, SacreBLEU (Standard & Indic-Tokenized), chrF, chrF++, METEOR, and TER.

In [ ]:
# ============================================================
# Evaluation + Save Metrics Permanently
# ============================================================
import os
import pandas as pd
import sacrebleu
import evaluate
import nltk
from indicnlp.tokenize import indic_tokenize

SAVE_DIR = "/content/drive/MyDrive/GoogleTranslate_Results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. SacreBLEU (Standard 13a Tokenizer)
# Note: sacrebleu expects references as a list of reference streams: [reference]
# ------------------------------------------------------------
sacrebleu_13a = sacrebleu.corpus_bleu(
    predictions,
    [reference],
    tokenize="13a"
).score

# ------------------------------------------------------------
# 2. Indic-Tokenized SacreBLEU (AI4Bharat / IndicTrans2 Standard)
# Pre-tokenizing Malayalam with IndicNLP handles complex script & morphemes
# ------------------------------------------------------------
preds_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(p, lang="ml")) for p in predictions]
refs_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(r, lang="ml")) for r in reference]

sacrebleu_indic = sacrebleu.corpus_bleu(
    preds_indic_tok,
    [refs_indic_tok],
    tokenize="none"
).score

# ------------------------------------------------------------
# 3. SacreBLEU (FLORES-200 Tokenizer if supported)
# ------------------------------------------------------------
try:
    sacrebleu_flores = sacrebleu.corpus_bleu(
        predictions,
        [reference],
        tokenize="flores200"
    ).score
except Exception:
    try:
        sacrebleu_flores = sacrebleu.corpus_bleu(
            predictions,
            [reference],
            tokenize="flores101"
        ).score
    except Exception:
        sacrebleu_flores = None

# ------------------------------------------------------------
# 4. HuggingFace Evaluate BLEU
# evaluate.load('bleu') expects references as [[r1], [r2], ...]
# ------------------------------------------------------------
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[x] for x in reference]
)["bleu"]

# ------------------------------------------------------------
# 5. chrF and chrF++ (Character n-gram F-score; Primary for Indic MT)
# ------------------------------------------------------------
chrf_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=1
).score

chrfpp_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=2
).score

# ------------------------------------------------------------
# 6. TER (Translation Edit Rate)
# ------------------------------------------------------------
ter_score = sacrebleu.corpus_ter(
    predictions,
    [reference]
).score

# ------------------------------------------------------------
# 7. METEOR
# ------------------------------------------------------------
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=reference
)["meteor"]

# ============================================================
# Print Summary
# ============================================================
print("\n" + "=" * 65)
print("     GOOGLE TRANSLATE EVALUATION RESULTS ON ANNOTATED DATASET")
print("=" * 65)
print(f"Dataset Sentences             : {len(predictions)}")
print("-" * 65)
print(f"SacreBLEU (Standard '13a')    : {sacrebleu_13a:.2f}")
print(f"SacreBLEU (Indic-Tokenized)   : {sacrebleu_indic:.2f}  <-- Recommended for Indic MT")
if sacrebleu_flores is not None:
    print(f"SacreBLEU (FLORES)            : {sacrebleu_flores:.2f}")
print(f"HuggingFace BLEU              : {bleu_score:.4f}")
print(f"chrF                          : {chrf_score:.2f}")
print(f"chrF++ (word_order=2)         : {chrfpp_score:.2f}  <-- Primary Morphological Metric")
print(f"METEOR                        : {meteor_score:.4f}")
print(f"TER (Lower is better)         : {ter_score:.2f}")
# ============================================================
# Save Metrics to Files
# ============================================================
metrics_dict = {
    "Model": ["Google Translate (deep-translator)"],
    "Dataset": ["Annotated Data - Filtered Dataset.csv"],
    "Sentence_Count": [len(predictions)],
    "SacreBLEU_13a": [round(sacrebleu_13a, 2)],
    "SacreBLEU_Indic_Tokenized": [round(sacrebleu_indic, 2)],
    "SacreBLEU_FLORES": [round(sacrebleu_flores, 2) if sacrebleu_flores is not None else None],
    "HF_BLEU": [round(bleu_score, 4)],
    "chrF": [round(chrf_score, 2)],
    "chrF++": [round(chrfpp_score, 2)],
    "METEOR": [round(meteor_score, 4)],
    "TER": [round(ter_score, 2)],
}

metrics_df = pd.DataFrame(metrics_dict)
metrics_csv_drive = os.path.join(SAVE_DIR, "google_translate_annotated_metrics.csv")
metrics_txt_drive = os.path.join(SAVE_DIR, "google_translate_annotated_metrics.txt")

metrics_df.to_csv(metrics_csv_drive, index=False)
metrics_df.to_csv("/content/google_translate_annotated_metrics.csv", index=False)

with open(metrics_txt_drive, "w", encoding="utf-8") as f:
    for col in metrics_df.columns:
        f.write(f"{col}: {metrics_df[col][0]}\n")

with open("/content/google_translate_annotated_metrics.txt", "w", encoding="utf-8") as f:
    for col in metrics_df.columns:
        f.write(f"{col}: {metrics_df[col][0]}\n")

print(f"\nEvaluation metrics saved to:\n1. {metrics_csv_drive}\n2. {metrics_txt_drive}\n3. /content/google_translate_annotated_metrics.csv")
display(metrics_df)


## 4. Benchmark Comparison Across All Models (IndicTrans2 vs BhashaVerse vs Google Translate)
Loads and presents a side-by-side comparison table of all three translation systems on `Annotated Data - Filtered Dataset.csv`.

In [ ]:
# Side-by-Side Model Comparison (IndicTrans2 vs BhashaVerse vs Google Translate)
import os
import pandas as pd

MODEL_PATHS = {
    "IndicTrans2": "/content/drive/MyDrive/IndicTrans2_Results/indictrans2_annotated_metrics.csv",
    "BhashaVerse": "/content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered/bhashaverse_annotated_metrics.csv",
    "Google Translate": "/content/drive/MyDrive/GoogleTranslate_Results/google_translate_annotated_metrics.csv",
}

dfs = []
for model_name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        m_df = pd.read_csv(path)
        m_df["Model"] = model_name
        dfs.append(m_df)

if dfs:
    comparison_df = pd.concat(dfs, ignore_index=True)
    comparison_df = comparison_df.sort_values(by="chrF++", ascending=False).reset_index(drop=True)
    print("=" * 80)
    print("      CROSS-MODEL TRANSLATION BENCHMARK (Annotated Filtered Dataset)")
    print("=" * 80)
    display_cols = [c for c in [
        "Model", "SacreBLEU_13a", "SacreBLEU_Indic_Tokenized",
        "chrF++", "METEOR", "TER"
    ] if c in comparison_df.columns]
    display(comparison_df[display_cols])

    comparison_df.to_csv("/content/drive/MyDrive/MT_All_Models_Comparison.csv", index=False)
    print(f"\nComparison table saved to: /content/drive/MyDrive/MT_All_Models_Comparison.csv")
else:
    print("Run the models to populate the benchmark comparison table.")


## 5. Compute COMET Metric (Unbabel/wmt22-comet-da)
This cell installs `unbabel-comet`, runs neural cross-lingual evaluation (`Unbabel/wmt22-comet-da`) on the predictions, and permanently updates `google_translate_annotated_metrics.csv` and `google_translate_annotated_metrics.txt`.

In [ ]:
# ============================================================
# Compute COMET Metric (Unbabel/wmt22-comet-da) & Update Files
# ============================================================
%%capture
!pip install -q unbabel-comet

import os
import json
import torch
import pandas as pd
from comet import download_model, load_from_checkpoint

# 1. Locate Predictions File
pred_candidates = [
    "/content/drive/MyDrive/Inference/GoogleTranslate_Results/google_translate_annotated_predictions.csv",
    "/content/drive/MyDrive/GoogleTranslate_Results/google_translate_annotated_predictions.csv",
    "/content/drive/MyDrive/Inference/GoogleTranslate_Results/google_translate_checkpoint.csv",
    "/content/google_translate_annotated_predictions.csv",
    os.path.join(os.getcwd(), "Inference", "GoogleTranslate_Results", "google_translate_annotated_predictions.csv"),
    os.path.join(os.getcwd(), "GoogleTranslate_Results", "google_translate_annotated_predictions.csv"),
    os.path.join(os.getcwd(), "google_translate_checkpoint.csv"),
]

pred_csv = next((p for p in pred_candidates if os.path.exists(p)), None)

if pred_csv and os.path.exists(pred_csv):
    print(f"Loading predictions from: {pred_csv}")
    df_eval = pd.read_csv(pred_csv, encoding="utf-8-sig")
    en_col = next((c for c in df_eval.columns if "english" in c.lower()), df_eval.columns[0])
    ref_col = next((c for c in df_eval.columns if "malayalam" in c.lower() and "google" not in c.lower() and "pred" not in c.lower()), df_eval.columns[1])
    mt_col = next((c for c in df_eval.columns if "google" in c.lower() or "pred" in c.lower()), df_eval.columns[-1])

    src_list = df_eval[en_col].astype(str).str.strip().tolist()
    ref_list = df_eval[ref_col].astype(str).str.strip().tolist()
    mt_list = df_eval[mt_col].astype(str).str.strip().tolist()
elif "en_sents" in globals() and "predictions" in globals() and "reference" in globals():
    src_list = en_sents
    mt_list = predictions
    ref_list = reference
    pred_csv = "/content/drive/MyDrive/Inference/GoogleTranslate_Results/google_translate_annotated_predictions.csv"
else:
    raise FileNotFoundError("Could not locate predictions CSV file to compute COMET.")

print(f"Evaluating {len(src_list)} sentence pairs with COMET...")

# 2. Load COMET model
print("Downloading and loading model: Unbabel/wmt22-comet-da...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# 3. Run Inference
comet_data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src_list, mt_list, ref_list)]
comet_output = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
comet_score = round(float(comet_output.system_score), 4)

print("\n" + "=" * 60)
print(f"  GOOGLE TRANSLATE COMET SCORE (wmt22-comet-da): {comet_score:.4f}")
print("=" * 60)

# 4. Update Metrics CSV & TXT permanently across all result paths
save_dirs = [
    os.path.dirname(pred_csv) if pred_csv else None,
    "/content/drive/MyDrive/Inference/GoogleTranslate_Results",
    "/content/drive/MyDrive/GoogleTranslate_Results",
    "/content"
]

updated_df = None
for sdir in [d for d in save_dirs if d and os.path.exists(d)]:
    m_csv = os.path.join(sdir, "google_translate_annotated_metrics.csv")
    m_txt = os.path.join(sdir, "google_translate_annotated_metrics.txt")
    if os.path.exists(m_csv):
        df_m = pd.read_csv(m_csv)
        df_m["COMET"] = comet_score
        df_m.to_csv(m_csv, index=False)
        updated_df = df_m
        print(f"Updated {m_csv}")
    if os.path.exists(m_txt):
        with open(m_txt, "r", encoding="utf-8") as f:
            lines = f.readlines()
        newlines = [l for l in lines if not l.startswith("COMET:")]
        newlines.append(f"COMET: {comet_score}\n")
        with open(m_txt, "w", encoding="utf-8") as f:
            f.writelines(newlines)
        print(f"Updated {m_txt}")

if updated_df is not None:
    display(updated_df)
